# BTXRD Anatomy-Aware WSSS — Region-Conditioned Pipeline (parallel to thesis_final.ipynb)

This notebook runs the **anatomy-matched contrastive learning** pipeline on branch `pipeline-anatomy-contrastive`,
in parallel with (and without modifying) `thesis_final.ipynb`'s standard `btxrd_best`/`btxrd_hybrid` run.

```text
tumor_type + anatomy_region (upper limb / lower limb / pelvis, image-level labels only)
 -> DenseNet121 shared backbone
    -> tumor-type head (10-class CE, weighted)
    -> anatomy-region head (3-class CE, weight 0.2)
    -> region-conditioned tumor head (3x binary BCE, weight 0.1)
    -> anatomy-matched contrastive loss (weight 0.05, 3-epoch warmup)
 -> anatomy-conditioned LayerCAM: S = (z_c - z_normal) + beta*(z_tumor,r - z_normal,r), beta=0.5
    -> multi-scale (256/320/original) + horizontal-flip ensemble of THIS anatomy-conditioned CAM
 -> SAM ViT-B prompt ensemble (box + points, same recipe as btxrd_best)
 -> simple_hybrid/coverage_mass_sam selection + cross-CAM (global vs anatomy) consistency bonus
 -> pseudo-mask + 4-label confidence map (foreground-confident/uncertain, background, boundary-uncertain)
 -> U-Net: boundary-ignore (confidence-weighted loss) + weak/strong consistency regularization + ReduceLROnPlateau
```

Every stage calls the repository's own scripts (`train_classifier.py`, `generate_pseudo_masks.py`,
`train_segmentation.py`) with the anatomy-specific flags added during this design's implementation --
nothing here reimplements pipeline logic in notebook code. Ground-truth polygons are only opened in the
final evaluation cells (section 9), never during classifier/CAM/SAM/pseudo-mask/U-Net training or selection.

See `config.py`'s `BtxrdAnatomyPipelineConfig`/`BTXRD_ANATOMY_PIPELINE` for the fixed loss weights this
profile pins (0.2/0.1/0.05 classifier weights, 3-epoch contrastive warmup, beta=0.5, U-Net consistency=0.1).


## Execution order and switches

1. Repository checkout (branch `pipeline-anatomy-contrastive`) and environment audit
2. Dataset resolution + anatomy-region label quality check (upper limb/lower limb/pelvis)
3. Classifier training with `--pipeline-profile btxrd_anatomy` (3 heads + contrastive loss)
4. Training-curve/CAM-preview inspection
5. Val-split pseudo-mask generation with `--cam-anatomy-conditioned --save-confidence-map`
6. Val-split pseudo-mask evaluation against GT polygons (diagnostic, does not affect selection)
7. Train-split pseudo-mask + confidence-map generation (needed for U-Net training)
8. U-Net training with `--boundary-ignore-loss --consistency-weight --lr-scheduler-plateau`
9. Final GT evaluation of the trained U-Net (opens ground-truth polygons for the first time)

Toggle switches live in Cell 1. No test-set tuning is performed anywhere in this notebook.


In [ ]:

# Cell 1 — paths and run switches
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

NOTEBOOK_ROOT = Path.cwd()
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
DEFAULT_WORKING = KAGGLE_WORKING if KAGGLE_WORKING.exists() else NOTEBOOK_ROOT / "notebook_runs"
REPO_URL = os.environ.get("BTXRD_REPO_URL", "https://github.com/itsthang333/Thesis.git")
# This notebook is pinned to the anatomy-aware branch -- thesis_final.ipynb
# stays on 'pipeline' and is never touched by this notebook.
GIT_BRANCH = os.environ.get("BTXRD_GIT_BRANCH", "pipeline-anatomy-contrastive")
DATASET_OVERRIDE = os.environ.get("BTXRD_ROOT", "")

# Fixed for this notebook: btxrd_anatomy pins the classifier's anatomy-region
# head, region-conditioned tumor head, and contrastive loss to their design-doc
# weights (0.2/0.1/0.05, see config.py's BtxrdAnatomyPipelineConfig). Unlike
# thesis_final.ipynb there is no PIPELINE_PROFILE switch here -- this notebook
# exists specifically to run btxrd_anatomy, not to A/B it against the others.
PIPELINE_PROFILE = "btxrd_anatomy"

if (NOTEBOOK_ROOT / "project").exists():
    PROJECT_PARENT = NOTEBOOK_ROOT
else:
    PROJECT_PARENT = KAGGLE_WORKING / "Thesis"
PROJECT_DIR = PROJECT_PARENT / "project"

OUTPUT_ROOT = Path(os.environ.get("BTXRD_OUTPUT", str(DEFAULT_WORKING / "btxrd_anatomy_pipeline")))
CLASSIFIER_OUTPUT = OUTPUT_ROOT / "classifier_btxrd_anatomy"
PREDICTED_OUTPUT = OUTPUT_ROOT / "pseudo_predicted_val"
TRAIN_PREDICTED_OUTPUT = OUTPUT_ROOT / "pseudo_predicted_train"
EVAL_OUTPUT = OUTPUT_ROOT / "evaluations"
UNET_OUTPUT = OUTPUT_ROOT / "unet_anatomy"
DEFAULT_SAM = NOTEBOOK_ROOT / "sam_vit_b_01ec64.pth" if (NOTEBOOK_ROOT / "sam_vit_b_01ec64.pth").exists() else OUTPUT_ROOT / "sam_vit_b_01ec64.pth"
SAM_CHECKPOINT = Path(os.environ.get("SAM_CHECKPOINT", str(DEFAULT_SAM)))

IMAGE_SIZE = 320
SAM_IMAGE_SIZE = 512
NUM_WORKERS = int(os.environ.get("BTXRD_NUM_WORKERS", "2"))

# CAM anatomy-conditioning knobs (see models/layercam.py's
# cam_for_anatomy_conditioned_score and config.py's BTXRD_ANATOMY_PIPELINE).
CAM_ANATOMY_BETA = 0.5
CAM_ANATOMY_WEIGHT = 1.0
ANATOMY_CONSISTENCY_WEIGHT = 0.2  # mask-selection cross-CAM agreement bonus
CAM_MULTISCALE_SIZES = "256,320"  # combined with the original size below
CAM_TTA_FLIP = True

# U-Net anatomy-aware training knobs (see train_segmentation.py).
UNET_EPOCHS = 300
UNET_EARLY_STOP_PATIENCE = 6
UNET_MIN_DELTA = 0.002
UNET_CONSISTENCY_WEIGHT = 0.1
UNET_CONFIDENCE_UNCERTAIN_WEIGHT = 0.5

# SMOKE_TEST=True runs the whole notebook end-to-end on a tiny slice, to catch
# runtime errors (CLI typos, missing files, shape mismatches under real data)
# in minutes instead of committing to the full multi-hour run below. It does
# NOT change which flags are passed (--pipeline-profile btxrd_anatomy,
# --cam-anatomy-conditioned, --boundary-ignore-loss, etc. are all identical
# to the full run) -- only how MUCH data/how many epochs each stage uses.
# NOTE: --pipeline-profile btxrd_anatomy locks --epochs to 25 (it raises if
# you pass a different value explicitly, see train_classifier.py's
# apply_pipeline_profile/require_or_set) -- classifier epochs are NOT
# reduced here, since the classifier is also the fastest of the three stages
# on 320px images. Only --max-images (pseudo-mask/SAM stage) and U-Net epochs
# are cut, since those are what actually dominate wall-clock time on a full
# split. A smoke-test PASS does not mean the pipeline actually improves
# Dice, only that it runs to completion without crashing -- still set
# SMOKE_TEST=False and budget for the full run to get a real answer.
SMOKE_TEST = False

if SMOKE_TEST:
    MAX_IMAGES_PER_SPLIT = 20               # instead of --process-all (full split)
    UNET_EPOCHS = 3
    UNET_EARLY_STOP_PATIENCE = 0            # disabled -- 3 epochs is already the whole run
    OUTPUT_ROOT = Path(os.environ.get("BTXRD_OUTPUT", str(DEFAULT_WORKING / "btxrd_anatomy_pipeline_smoke")))
    CLASSIFIER_OUTPUT = OUTPUT_ROOT / "classifier_btxrd_anatomy"
    PREDICTED_OUTPUT = OUTPUT_ROOT / "pseudo_predicted_val"
    TRAIN_PREDICTED_OUTPUT = OUTPUT_ROOT / "pseudo_predicted_train"
    EVAL_OUTPUT = OUTPUT_ROOT / "evaluations"
    UNET_OUTPUT = OUTPUT_ROOT / "unet_anatomy"
else:
    MAX_IMAGES_PER_SPLIT = None             # use --process-all (full split)

INSTALL_DEPENDENCIES = True
RUN_TRAIN_CLASSIFIER = True
RUN_FULL_PREDICTED_VAL = True
RUN_EVALUATE_PREDICTED_VAL = True
RUN_FULL_PREDICTED_TRAIN = True
RUN_TRAIN_UNET = True
RUN_FINAL_EVAL = True

for path in [OUTPUT_ROOT, CLASSIFIER_OUTPUT, PREDICTED_OUTPUT, TRAIN_PREDICTED_OUTPUT, EVAL_OUTPUT, UNET_OUTPUT]:
    path.mkdir(parents=True, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)
print("GIT_BRANCH:", GIT_BRANCH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("PIPELINE_PROFILE:", PIPELINE_PROFILE)
print("SMOKE_TEST:", SMOKE_TEST)


## 1. Repository checkout and environment audit

In [ ]:

# Cell 2 — checkout, imports, and streaming subprocess helper
if not PROJECT_DIR.exists():
    PROJECT_PARENT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(PROJECT_PARENT)], check=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def run_streaming(cmd, cwd=PROJECT_DIR, check=True):
    cmd = [str(x) for x in cmd]
    if cmd and Path(cmd[0]).name.lower().startswith("python"):
        cmd = [cmd[0], "-u", *cmd[1:]]
    header = "$ " + " ".join(shlex.quote(x) for x in cmd)
    print(header)
    # See thesis_final.ipynb's Cell 2 for the full rationale on why this
    # streams raw bytes with an incremental UTF-8 decoder instead of
    # text=True + IPython.display.clear_output: Kaggle's committed-run Logs
    # tab is a flat, append-only stream where clear_output() is a no-op, so
    # naive redraw-the-whole-log logic grows the log quadratically with
    # training length. This throttles in-progress '\r' snapshots to once
    # every few seconds and never replays history.
    import codecs
    import time

    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    process = subprocess.Popen(cmd, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    buffer = ""
    last_progress_print = 0.0
    min_progress_interval = 3.0
    for raw_byte in iter(lambda: process.stdout.read(1), b""):
        chunk = decoder.decode(raw_byte)
        if not chunk:
            continue
        if chunk == "\r":
            now = time.monotonic()
            if buffer and now - last_progress_print >= min_progress_interval:
                print(buffer)
                last_progress_print = now
            buffer = ""
        elif chunk == "\n":
            print(buffer)
            buffer = ""
        else:
            buffer += chunk
    if buffer:
        print(buffer)
    result = process.wait()
    if check and result:
        raise subprocess.CalledProcessError(result, cmd)
    return result

if INSTALL_DEPENDENCIES:
    run_streaming([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")])
    run_streaming([sys.executable, "-m", "pip", "install", "-q", "pandas", "openpyxl", "opencv-python"])
print("cwd:", Path.cwd())


In [ ]:

# Cell 2b — split a pseudo-mask generation run across BOTH T4 GPUs.
#
# generate_pseudo_masks.py itself is single-GPU (torch.device("cuda") picks
# one device, no nn.DataParallel/multi-process logic inside it) -- unlike
# train_segmentation.py's --multi-gpu, there is no in-script way to use both
# GPUs for this stage. Since each image's CAM/SAM pass is independent of
# every other image, the parallelism is applied at the PROCESS level instead:
# split the split's image list in half, run one generate_pseudo_masks.py
# subprocess per GPU (pinned via CUDA_VISIBLE_DEVICES, since
# --classifier-device only accepts "auto"/"cpu"/"cuda", not "cuda:1" --
# CUDA_VISIBLE_DEVICES remaps whichever GPU it names to appear as cuda:0
# inside that subprocess), then merge each GPU's own --output-dir into one.
#
# Each GPU writes to its OWN --output-dir (not a shared one): sharing one
# --output-dir across two concurrent processes would race on
# write_or_validate_run_metadata's read-then-write of run_metadata.json, and
# --image-list is itself recorded IN that metadata dict -- two processes
# with two different (half-split) --image-list values would see each
# other's metadata as a "protocol mismatch" and raise, since
# write_or_validate_run_metadata was designed to guard against exactly that
# kind of run-to-run inconsistency, not to recognize "these are two
# intentional halves of one split."
def write_image_list(dataset, split, image_size, list_path):
    from datasets.factory import build_classification_dataset as _build
    ds = _build(dataset, root=BTXRD_ROOT, split=split, target_columns=["tumor_type"], image_size=image_size)
    image_ids = [str(sample["image_id"]) for sample in ds.samples]
    list_path.write_text("\n".join(image_ids) + "\n", encoding="utf-8")
    return len(image_ids)

def run_dual_gpu_pseudo_masks(base_cmd, output_dir: Path, dataset: str, split: str, image_size: int, max_images=None):
    """base_cmd must NOT include --output-dir, --image-list, --classifier-device,
    --sam-device, --process-all, or --max-images -- this function adds all of
    them (max_images only on the single-GPU fallback path; the dual-GPU path
    always splits the FULL split in half regardless of max_images, since
    SMOKE_TEST's point is to exercise the real dual-GPU merge code path too,
    not skip it). Merges both halves' masks/overlays/confidence/ PNGs into
    `output_dir` afterward, and writes a combined run_metadata.json (recording
    BOTH half-output-dirs for traceability) plus a combined
    skipped_low_confidence.txt so downstream cells that read `output_dir`
    directly (Cell 12/13/14/17) see one unified result, exactly as if a
    single-GPU --process-all run had produced it.
    """
    num_gpus = torch.cuda.device_count()
    if num_gpus < 2:
        print(f"Only {num_gpus} GPU(s) visible -- running single-GPU (no split).")
        cmd = base_cmd + ["--output-dir", str(output_dir)]
        cmd += ["--process-all"] if max_images is None else ["--max-images", str(max_images)]
        run_streaming(cmd)
        return

    gpu_output_dirs = [output_dir.parent / f"{output_dir.name}_gpu{i}" for i in range(2)]
    gpu_list_paths = [output_dir.parent / f"{output_dir.name}_gpu{i}_images.txt" for i in range(2)]
    total_images = write_image_list(dataset, split, image_size, output_dir.parent / f"{output_dir.name}_all_images.txt")
    all_image_ids = (output_dir.parent / f"{output_dir.name}_all_images.txt").read_text().splitlines()
    if max_images is not None:
        # SMOKE_TEST: cap the TOTAL before splitting, so both halves stay
        # small -- still exercises the real dual-GPU split/merge code path,
        # just on a tiny slice instead of the full split.
        all_image_ids = all_image_ids[:max_images]
    halves = [all_image_ids[: len(all_image_ids) // 2], all_image_ids[len(all_image_ids) // 2 :]]
    for half, list_path in zip(halves, gpu_list_paths):
        list_path.write_text("\n".join(half) + "\n", encoding="utf-8")
    print(f"Split {total_images} images: GPU0={len(halves[0])}, GPU1={len(halves[1])}")

    processes = []
    log_files = []
    for gpu_index in range(2):
        gpu_output_dirs[gpu_index].mkdir(parents=True, exist_ok=True)
        cmd = [str(x) for x in base_cmd] + [
            "--output-dir", str(gpu_output_dirs[gpu_index]),
            "--image-list", str(gpu_list_paths[gpu_index]),
            "--classifier-device", "cuda", "--sam-device", "cuda",
        ]
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu_index))
        log_path = output_dir.parent / f"{output_dir.name}_gpu{gpu_index}.log"
        log_handle = log_path.open("w", encoding="utf-8")
        log_files.append(log_handle)
        header = "$ CUDA_VISIBLE_DEVICES=" + str(gpu_index) + " " + " ".join(shlex.quote(x) for x in cmd)
        print(header)
        process = subprocess.Popen(
            [str(x) for x in cmd], cwd=str(PROJECT_DIR), env=env,
            stdout=log_handle, stderr=subprocess.STDOUT,
        )
        processes.append(process)

    import time
    while any(p.poll() is None for p in processes):
        time.sleep(15)
    for handle in log_files:
        handle.close()
    for gpu_index, process in enumerate(processes):
        # Tail each GPU's log so failures are visible in the notebook's own
        # output, not just in the *_gpu{N}.log file on disk.
        log_path = output_dir.parent / f"{output_dir.name}_gpu{gpu_index}.log"
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
        print(f"--- GPU{gpu_index} log tail ---")
        print("\n".join(tail))
    for gpu_index, process in enumerate(processes):
        if process.returncode != 0:
            raise RuntimeError(f"GPU{gpu_index} generate_pseudo_masks.py exited with code {process.returncode}; see {output_dir.parent / f'{output_dir.name}_gpu{gpu_index}.log'}")

    # Merge: masks/, overlays/, confidence/ are disjoint per-image PNG sets
    # (each image was processed by exactly one GPU), so this is a plain
    # union copy, never an overwrite of the same filename from both halves.
    output_dir.mkdir(parents=True, exist_ok=True)
    for subdir_name in ["masks", "overlays", "confidence"]:
        merged_subdir = output_dir / subdir_name
        merged_subdir.mkdir(parents=True, exist_ok=True)
        for gpu_output_dir in gpu_output_dirs:
            source_subdir = gpu_output_dir / subdir_name
            if not source_subdir.exists():
                continue
            for file_path in source_subdir.glob("*"):
                shutil.copy2(file_path, merged_subdir / file_path.name)

    # Merge skipped_low_confidence.txt (concatenate, both halves' skip lists).
    skipped_lines = []
    for gpu_output_dir in gpu_output_dirs:
        skip_path = gpu_output_dir / "skipped_low_confidence.txt"
        if skip_path.exists():
            skipped_lines.extend(skip_path.read_text(encoding="utf-8").splitlines())
    if skipped_lines:
        (output_dir / "skipped_low_confidence.txt").write_text("\n".join(skipped_lines) + "\n", encoding="utf-8")

    # Merge run_metadata.json: take GPU0's as the base (both halves ran with
    # identical flags except --image-list/--output-dir/--classifier-device,
    # which are dropped here since they're per-GPU artifacts, not part of
    # the shared recipe Cell 12's cam_anatomy_conditioned check reads).
    metadata_written = False
    for gpu_output_dir in gpu_output_dirs:
        metadata_path = gpu_output_dir / "run_metadata.json"
        if metadata_path.exists():
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            metadata.pop("image_list", None)
            metadata["dual_gpu_split"] = True
            (output_dir / "run_metadata.json").write_text(
                json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8"
            )
            metadata_written = True
            break
    if not metadata_written:
        print("WARNING: neither GPU half wrote a run_metadata.json -- check both *_gpuN.log files.")

    # Merge anatomy_region_unknown_fallback_count.txt (sum both halves).
    fallback_total = 0
    for gpu_output_dir in gpu_output_dirs:
        fallback_path = gpu_output_dir / "anatomy_region_unknown_fallback_count.txt"
        if fallback_path.exists():
            fallback_total += int(fallback_path.read_text().strip())
    if fallback_total > 0:
        (output_dir / "anatomy_region_unknown_fallback_count.txt").write_text(f"{fallback_total}\n", encoding="utf-8")

    print(f"Merged {len(list((output_dir / 'masks').glob('*.png')))} masks into {output_dir}")


In [ ]:

# Cell 3 — hardware/runtime audit
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(index, torch.cuda.get_device_name(index), round(props.total_memory / 2**30, 2), "GiB")


## 2. Dataset resolution and anatomy-region label quality check

In [ ]:

# Cell 4 — locate BTXRD and inspect records
from datasets.btxrd import (
    ANATOMY_REGION_COLUMNS, TUMOR_TYPE_CLASS_NAMES, load_btxrd_records,
    resolve_btxrd_root, split_btxrd_records,
)
from datasets.factory import build_classification_dataset

def find_btxrd_root(base: Path):
    candidates = [base, *base.glob("*"), *base.glob("*/*")] if base and base.exists() else []
    for candidate in candidates:
        try:
            return resolve_btxrd_root(candidate)
        except FileNotFoundError:
            pass
    return None

# Hardcoded to this Kaggle dataset mount -- skips the auto-detection glob
# below entirely. Fall back to auto-detection only if this exact path
# doesn't exist (e.g. running outside this specific Kaggle input mount).
BTXRD_ROOT = Path("/kaggle/input/datasets/wanwin/data-btxrd/BTXRD")
if not BTXRD_ROOT.exists():
    search_root = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else (KAGGLE_INPUT if KAGGLE_INPUT.exists() else NOTEBOOK_ROOT)
    BTXRD_ROOT = find_btxrd_root(search_root)
if BTXRD_ROOT is None or not BTXRD_ROOT.exists():
    raise FileNotFoundError("BTXRD not found. Set BTXRD_ROOT or attach images/, Annotations/, dataset.csv/xlsx.")
BTXRD_ROOT = resolve_btxrd_root(BTXRD_ROOT)
records = load_btxrd_records(BTXRD_ROOT)
print("BTXRD_ROOT:", BTXRD_ROOT)
print("records:", len(records))
print("ANATOMY_REGION_COLUMNS:", ANATOMY_REGION_COLUMNS)


In [ ]:

# Cell 5 — anatomy-region label quality gate: run BEFORE any anatomy-aware training.
# See tools/check_anatomy_region_labels.py for the standalone CLI version of this check;
# this cell reuses the same logic inline so the notebook fails fast with a clear message
# instead of training_classifier.py silently building a checkpoint no one can trust.
unknown = [r for r in records if r["anatomy_region"] == -1]
print(f"Records with unknown anatomy_region (-1): {len(unknown)}/{len(records)}")

region_rows = []
for i, region_name in enumerate(ANATOMY_REGION_COLUMNS):
    region_records = [r for r in records if r["anatomy_region"] == i]
    normal_count = sum(1 for r in region_records if r["tumor"] == 0)
    tumor_count = sum(1 for r in region_records if r["tumor"] == 1)
    region_rows.append({"region": region_name, "normal": normal_count, "tumor": tumor_count, "total": len(region_records)})
    if normal_count == 0 or tumor_count == 0:
        raise RuntimeError(
            f"Region '{region_name}' has no {'normal' if normal_count == 0 else 'tumor'} images -- "
            "anatomy-matched contrastive learning/region-conditioned CAM cannot pair this region. "
            "Check dataset.csv's upper limb/lower limb/pelvis columns before proceeding."
        )
display(pd.DataFrame(region_rows))

unknown_fraction = len(unknown) / len(records)
if unknown_fraction > 0.05:
    raise RuntimeError(
        f"{unknown_fraction:.1%} of records have unknown anatomy_region -- this is far above the "
        "0% found on the real BTXRD dataset.csv during this project's own verification. Check "
        "--ram-root points at the intended dataset.csv before training the anatomy-aware classifier."
    )
print("Anatomy-region labels look clean -- safe to proceed with btxrd_anatomy.")


In [ ]:

# Cell 6 — split/class distribution (same stratified 80/10/10 split as thesis_final.ipynb)
def split_frame(name):
    rows = split_btxrd_records(records, split=name, seed=42)
    frame = pd.DataFrame(rows)
    frame["tumor_type_name"] = frame["tumor_type"].map(dict(enumerate(TUMOR_TYPE_CLASS_NAMES)))
    return frame

split_frames = {name: split_frame(name) for name in ["train", "val", "test"]}
for name, frame in split_frames.items():
    print(f"{name}: n={len(frame)} tumor={int((frame.tumor_type > 0).sum())} normal={int((frame.tumor_type == 0).sum())}")

classification_ds = build_classification_dataset(
    "btxrd", root=BTXRD_ROOT, split="val", target_columns=["tumor_type"], image_size=IMAGE_SIZE,
)
assert classification_ds.target_columns == ["tumor_type"]
print("Generation will use image-level target only:", classification_ds.target_columns)
print("Segmentation masks/polygons remain diagnostic/evaluation-only objects throughout training.")


## 3. SAM checkpoint and anatomy-aware classifier training

In [ ]:

# Cell 7 — SAM checkpoint
SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not SAM_CHECKPOINT.exists():
    import urllib.request
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", str(SAM_CHECKPOINT))
print("SAM:", SAM_CHECKPOINT, "GiB:", round(SAM_CHECKPOINT.stat().st_size / 2**30, 3))


In [ ]:

# Cell 8 — train classifier with --pipeline-profile btxrd_anatomy.
# This single profile flag pins ALL of: 25 epochs, PuzzleCAM+Teacher-Student
# (inherited from btxrd_hybrid), PLUS the anatomy-region head (weight 0.2),
# region-conditioned tumor head (weight 0.1), and anatomy-matched contrastive
# loss (weight 0.05, 3-epoch warmup) -- see config.py's BtxrdAnatomyPipelineConfig.
# No anatomy-specific CLI flags need to be passed explicitly; the profile sets
# them and train_classifier.py rejects any attempt to override them differently.
CLASSIFIER_CHECKPOINT = CLASSIFIER_OUTPUT / "best_classifier.pt"
classifier_cmd = [
    sys.executable, "train_classifier.py", "--dataset", "btxrd", "--pipeline-profile", PIPELINE_PROFILE,
    "--ram-root", str(BTXRD_ROOT), "--num-workers", str(NUM_WORKERS),
    "--save-cam-epochs", "2,5,10,15,20,25", "--cam-preview-count", "4",
    "--output-dir", str(CLASSIFIER_OUTPUT),
]
print(' '.join(classifier_cmd))
if RUN_TRAIN_CLASSIFIER:
    run_streaming(classifier_cmd)
else:
    print("RUN_TRAIN_CLASSIFIER=False")
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(CLASSIFIER_CHECKPOINT)


In [ ]:

# Cell 9 — training curves and checkpoint audit, including the anatomy-specific fields
training_log = CLASSIFIER_OUTPUT / "training_log.csv"
if training_log.exists():
    train_df = pd.read_csv(training_log)
    display(train_df.tail(10))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    train_df[["train_loss", "val_loss"]].plot(ax=axes[0], marker="o", title="CE loss")
    train_df[["train_f1", "val_f1"]].plot(ax=axes[1], marker="o", title="macro-F1")
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

state = torch.load(CLASSIFIER_CHECKPOINT, map_location="cpu")
expected = {
    "task": "single-label", "target_columns": ["tumor_type"], "num_classes": 10,
    "normalization": "imagenet", "pipeline_profile": "btxrd_anatomy",
}
for key, value in expected.items():
    print(key, state.get(key), "expected", value)
    assert state.get(key) == value
# These two fields are what generate_pseudo_masks.py's load_classifier() reads
# to decide whether it's safe to reconstruct the model with a region-tumor
# head before --cam-anatomy-conditioned can be used (see train_classifier.py's
# save_checkpoint and this project's own audit of the checkpoint round-trip).
print("num_anatomy_regions:", state.get("num_anatomy_regions"), "(expected 3)")
print("region_conditioned_tumor_head:", state.get("region_conditioned_tumor_head"), "(expected True)")
assert state.get("num_anatomy_regions") == 3
assert state.get("region_conditioned_tumor_head") is True
print("checkpoint:", CLASSIFIER_CHECKPOINT)


In [ ]:

# Cell 10 — CAM snapshots across epochs (same viewer as thesis_final.ipynb)
from collections import defaultdict
import re
cam_dir = CLASSIFIER_OUTPUT / "cam_preview"
by_sample = defaultdict(list)
for path in sorted(cam_dir.glob("cam_epoch*.png")) if cam_dir.exists() else []:
    match = re.match(r"cam_epoch(\d+)_(.+)\.png", path.name)
    if match: by_sample[match.group(2)].append((int(match.group(1)), path))
if by_sample:
    rows = sorted(by_sample.items()); ncols = max(len(x) for _, x in rows)
    fig, axes = plt.subplots(len(rows), ncols, figsize=(3.2*ncols, 3.2*len(rows)), squeeze=False)
    for r, (stem, entries) in enumerate(rows):
        for c, (epoch, path) in enumerate(sorted(entries)):
            axes[r, c].imshow(Image.open(path)); axes[r, c].set_title(f"{stem}\nepoch {epoch}"); axes[r, c].axis("off")
        for c in range(len(entries), ncols): axes[r, c].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No CAM snapshots found.")


## 4. Val-split pseudo-mask generation with anatomy-conditioned CAM

In [ ]:

# Cell 11 — full validation pseudo-masks with --cam-anatomy-conditioned + multi-view ensemble
# + cross-CAM consistency selection bonus + confidence map. All anatomy-specific flags are
# explicit here (unlike the classifier, generate_pseudo_masks.py's --pipeline-profile btxrd_best
# only freezes the non-anatomy CAM/SAM/selection recipe -- these flags layer on top of it).
#
# --output-dir/--image-list/--process-all/--max-images/--classifier-device/--sam-device are
# intentionally NOT included here -- run_dual_gpu_pseudo_masks (Cell 2b) adds all of them,
# splitting this stage across both T4 GPUs via CUDA_VISIBLE_DEVICES (generate_pseudo_masks.py
# itself has no multi-GPU support, unlike train_segmentation.py's --multi-gpu).
PREDICTED_CMD = [
    sys.executable, "generate_pseudo_masks.py", "--dataset", "btxrd", "--pipeline-profile", "btxrd_best",
    "--ram-root", str(BTXRD_ROOT), "--split", "val", "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--evaluate-prompt-quality",
    "--save-visuals-limit", "10",
    "--cam-anatomy-conditioned", "--cam-anatomy-beta", str(CAM_ANATOMY_BETA),
    "--cam-anatomy-weight", str(CAM_ANATOMY_WEIGHT),
    "--cam-multiscale-sizes", CAM_MULTISCALE_SIZES,
    "--anatomy-consistency-weight", str(ANATOMY_CONSISTENCY_WEIGHT),
    "--save-confidence-map",
]
if CAM_TTA_FLIP:
    PREDICTED_CMD.append("--cam-tta-flip")
print(' '.join(PREDICTED_CMD))
if RUN_FULL_PREDICTED_VAL:
    run_dual_gpu_pseudo_masks(
        PREDICTED_CMD, PREDICTED_OUTPUT, dataset="btxrd", split="val", image_size=IMAGE_SIZE,
        max_images=MAX_IMAGES_PER_SPLIT,
    )
else:
    print("RUN_FULL_PREDICTED_VAL=False; command prepared only.")


In [ ]:

# Cell 12 — sanity-check the anatomy-conditioned CAM actually ran (not silently falling back)
run_metadata_path = PREDICTED_OUTPUT / "run_metadata.json"
if run_metadata_path.exists():
    run_metadata = json.loads(run_metadata_path.read_text())
    display(pd.Series(run_metadata).to_frame("value"))
    assert run_metadata.get("cam_anatomy_conditioned") is True

fallback_count_path = PREDICTED_OUTPUT / "anatomy_region_unknown_fallback_count.txt"
if fallback_count_path.exists():
    print("Images that fell back to the plain class-contrast CAM (unknown anatomy_region):",
          fallback_count_path.read_text().strip())
else:
    print("No anatomy_region fallback -- every non-normal image used the anatomy-conditioned CAM.")

confidence_dir = PREDICTED_OUTPUT / "confidence"
mask_dir = PREDICTED_OUTPUT / "masks"
print("masks:", len(list(mask_dir.glob('*.png'))) if mask_dir.exists() else 0)
print("confidence maps:", len(list(confidence_dir.glob('*.png'))) if confidence_dir.exists() else 0)


In [ ]:

# Cell 13 — visualize a few pseudo-masks + confidence maps side by side
from pseudo.mask_selection import (
    CONFIDENCE_BACKGROUND, CONFIDENCE_FOREGROUND_CONFIDENT,
    CONFIDENCE_FOREGROUND_UNCERTAIN, CONFIDENCE_BOUNDARY_UNCERTAIN,
)

CONFIDENCE_COLORS = {
    CONFIDENCE_BACKGROUND: (30, 30, 30),
    CONFIDENCE_FOREGROUND_CONFIDENT: (40, 200, 40),
    CONFIDENCE_FOREGROUND_UNCERTAIN: (230, 200, 40),
    CONFIDENCE_BOUNDARY_UNCERTAIN: (220, 60, 60),
}

def colorize_confidence(path):
    arr = np.array(Image.open(path))
    rgb = np.zeros((*arr.shape, 3), dtype=np.uint8)
    for label, color in CONFIDENCE_COLORS.items():
        rgb[arr == label] = color
    return rgb

sample_masks = sorted(mask_dir.glob("*.png"))[:4] if mask_dir.exists() else []
if sample_masks:
    fig, axes = plt.subplots(2, len(sample_masks), figsize=(3.2 * len(sample_masks), 6.4), squeeze=False)
    for c, mask_path in enumerate(sample_masks):
        axes[0, c].imshow(Image.open(mask_path), cmap="gray"); axes[0, c].set_title(mask_path.stem); axes[0, c].axis("off")
        confidence_path = confidence_dir / mask_path.name
        if confidence_path.exists():
            axes[1, c].imshow(colorize_confidence(confidence_path)); axes[1, c].axis("off")
        else:
            axes[1, c].axis("off")
    axes[0, 0].set_ylabel("pseudo-mask")
    plt.tight_layout(); plt.show()
    print("Confidence legend: gray=background, green=fg-confident, yellow=fg-uncertain, red=boundary-uncertain")
else:
    print("No masks found to visualize.")


## 5. Val-split pseudo-mask evaluation against GT polygons (diagnostic)

In [ ]:

# Cell 14 — evaluate predicted protocol against polygons. This is diagnostic ONLY --
# it does not feed back into classifier/CAM/SAM/selection/U-Net training or checkpoint
# selection anywhere in this notebook (see Cell 18's warning check for the same guarantee
# on the U-Net side).
PREDICTED_EVAL = EVAL_OUTPUT / "predicted.csv"
predicted_eval_cmd = [
    sys.executable, "evaluate_ramh1200_masks.py", "--dataset", "btxrd", "--ram-root", str(BTXRD_ROOT),
    "--split", "val", "--image-size", str(IMAGE_SIZE), "--pred-mask-root", str(PREDICTED_OUTPUT / "masks"),
    "--output-csv", str(PREDICTED_EVAL),
]
if RUN_EVALUATE_PREDICTED_VAL:
    run_streaming(predicted_eval_cmd)
    if PREDICTED_EVAL.exists():
        eval_df = pd.read_csv(PREDICTED_EVAL)
        display(eval_df.describe())
else:
    print("Evaluation command prepared:", predicted_eval_cmd)


## 6. Train-split pseudo-mask + confidence-map generation (for U-Net training)

In [ ]:

# Cell 15 — same anatomy-conditioned CAM recipe as Cell 11, applied to the train split,
# also split across both T4 GPUs via run_dual_gpu_pseudo_masks (see Cell 2b/Cell 11).
TRAIN_PREDICTED_CMD = [
    sys.executable, "generate_pseudo_masks.py", "--dataset", "btxrd", "--pipeline-profile", "btxrd_best",
    "--ram-root", str(BTXRD_ROOT), "--split", "train", "--classifier-checkpoint", str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint", str(SAM_CHECKPOINT), "--evaluate-prompt-quality",
    "--save-visuals-limit", "0",
    "--cam-anatomy-conditioned", "--cam-anatomy-beta", str(CAM_ANATOMY_BETA),
    "--cam-anatomy-weight", str(CAM_ANATOMY_WEIGHT),
    "--cam-multiscale-sizes", CAM_MULTISCALE_SIZES,
    "--anatomy-consistency-weight", str(ANATOMY_CONSISTENCY_WEIGHT),
    "--save-confidence-map",
]
if CAM_TTA_FLIP:
    TRAIN_PREDICTED_CMD.append("--cam-tta-flip")
print(' '.join(TRAIN_PREDICTED_CMD))
# SMOKE_TEST caps the train split at a larger slice than val (MAX_IMAGES_PER_SPLIT * 4):
# train_segmentation.py's DataLoader needs enough images left after the tumor/normal
# split for at least one full batch, and the train split skews toward more normal
# images than val -- a too-small slice here risks an empty-batch runtime error in
# Cell 16 that has nothing to do with the anatomy-conditioned CAM/pseudo-mask code
# actually being smoke-tested.
train_max_images = None if MAX_IMAGES_PER_SPLIT is None else MAX_IMAGES_PER_SPLIT * 4
if RUN_FULL_PREDICTED_TRAIN:
    run_dual_gpu_pseudo_masks(
        TRAIN_PREDICTED_CMD, TRAIN_PREDICTED_OUTPUT, dataset="btxrd", split="train", image_size=IMAGE_SIZE,
        max_images=train_max_images,
    )
else:
    print("RUN_FULL_PREDICTED_TRAIN=False; command prepared only.")


## 7. U-Net training: boundary-ignore + consistency + ReduceLROnPlateau

In [ ]:

# Cell 16 — train U-Net on the pipeline's own pseudo-masks + confidence maps.
# --val-pred-mask-root/--val-pred-confidence-root are BOTH set here (pointing at
# PREDICTED_OUTPUT from Cell 11), so best_unet.pt/early-stopping/LR-plateau are all
# selected using PSEUDO-MASK val Dice -- never ground-truth polygon Dice (see
# train_segmentation.py's WARNING check, which would otherwise fire if
# --val-pred-mask-root were left unset while --train-pred-mask-root is set).
unet_cmd = [
    sys.executable, "train_segmentation.py", "--dataset", "btxrd", "--ram-root", str(BTXRD_ROOT),
    "--train-split", "train", "--val-split", "val", "--image-size", str(IMAGE_SIZE),
    "--num-workers", str(NUM_WORKERS), "--epochs", str(UNET_EPOCHS),
    "--early-stop-patience", str(UNET_EARLY_STOP_PATIENCE), "--min-delta", str(UNET_MIN_DELTA),
    "--train-pred-mask-root", str(TRAIN_PREDICTED_OUTPUT / "masks"),
    "--val-pred-mask-root", str(PREDICTED_OUTPUT / "masks"),
    "--train-pred-confidence-root", str(TRAIN_PREDICTED_OUTPUT / "confidence"),
    "--val-pred-confidence-root", str(PREDICTED_OUTPUT / "confidence"),
    "--boundary-ignore-loss", "--confidence-weighted-loss",
    "--confidence-uncertain-weight", str(UNET_CONFIDENCE_UNCERTAIN_WEIGHT),
    "--consistency-weight", str(UNET_CONSISTENCY_WEIGHT),
    "--lr-scheduler-plateau", "--lr-scheduler-patience", "2", "--lr-scheduler-factor", "0.5",
    "--output-dir", str(UNET_OUTPUT), "--multi-gpu",
]
print(' '.join(unet_cmd))
if RUN_TRAIN_UNET:
    run_streaming(unet_cmd)
else:
    print("RUN_TRAIN_UNET=False; command prepared only.")


In [ ]:

# Cell 17 — U-Net training curves (train_dice/val_dice/consistency)
unet_log = UNET_OUTPUT / "training_log.csv"
if unet_log.exists():
    unet_df = pd.read_csv(unet_log)
    display(unet_df.tail(10))
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    unet_df[["train_dice", "val_dice"]].plot(ax=axes[0], marker="o", title="Dice")
    unet_df[["train_loss", "val_loss"]].plot(ax=axes[1], marker="o", title="loss")
    if "train_consistency" in unet_df.columns and UNET_CONSISTENCY_WEIGHT > 0:
        unet_df[["train_consistency"]].plot(ax=axes[2], marker="o", title="consistency loss")
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No U-Net training log found.")


## 8. Final evaluation against ground-truth polygons

This is the FIRST cell in this notebook that opens ground-truth polygons for anything other than a
diagnostic side-comparison -- checkpoint selection, early-stopping, and LR scheduling above were all
driven by pseudo-mask val Dice.

In [ ]:

# Cell 18 — evaluate the trained U-Net against real GT polygons (never seen during training/selection).
UNET_EVAL_CSV = EVAL_OUTPUT / "unet_anatomy.csv"
UNET_EVAL_JSON = EVAL_OUTPUT / "unet_anatomy_summary.json"
unet_eval_cmd = [
    sys.executable, "evaluate_unet.py", "--dataset", "btxrd", "--ram-root", str(BTXRD_ROOT),
    "--split", "val", "--checkpoint", str(UNET_OUTPUT / "best_unet.pt"),
    "--image-size", str(IMAGE_SIZE), "--output-csv", str(UNET_EVAL_CSV), "--output-json", str(UNET_EVAL_JSON),
]
print(' '.join(unet_eval_cmd))
if RUN_FINAL_EVAL:
    run_streaming(unet_eval_cmd)
    if UNET_EVAL_JSON.exists():
        display(pd.Series(json.loads(UNET_EVAL_JSON.read_text())).to_frame("value"))
else:
    print("Evaluation command prepared:", unet_eval_cmd)


In [ ]:

# Cell 19 — artifact manifest, for locating everything after the run (Kaggle Output tab / Drive upload).
manifest = {
    "pipeline_profile": PIPELINE_PROFILE,
    "classifier_checkpoint": str(CLASSIFIER_CHECKPOINT),
    "predicted_val_masks": str(PREDICTED_OUTPUT / "masks"),
    "predicted_val_confidence": str(PREDICTED_OUTPUT / "confidence"),
    "predicted_train_masks": str(TRAIN_PREDICTED_OUTPUT / "masks"),
    "predicted_train_confidence": str(TRAIN_PREDICTED_OUTPUT / "confidence"),
    "unet_best_checkpoint": str(UNET_OUTPUT / "best_unet.pt"),
    "unet_training_log": str(UNET_OUTPUT / "training_log.csv"),
    "final_eval_json": str(UNET_EVAL_JSON) if 'UNET_EVAL_JSON' in dir() else None,
}
(EVAL_OUTPUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(pd.Series(manifest).to_frame("path"))
print("\nAll outputs are under OUTPUT_ROOT:", OUTPUT_ROOT)
print("On Kaggle, download from the Output tab (not the Notebook tab) to get checkpoints/masks/CSVs.")


## Interpretation checklist

- **Anatomy-region label quality** (section 2) must pass with 0% unknown regions and both tumor/normal present
  in every region before trusting anything downstream.
- **Checkpoint fields** (Cell 9) confirm `num_anatomy_regions=3` and `region_conditioned_tumor_head=True` --
  if either is missing/False, `--cam-anatomy-conditioned` in Cell 11/15 would have raised immediately.
- **`run_metadata.json`'s `cam_anatomy_conditioned=true`** (Cell 12) combined with a LOW `anatomy_region_unknown_fallback_count`
  confirms the anatomy-conditioned CAM actually ran for most images, not silently falling back to the plain
  class-contrast CAM (a >50% fallback rate raises a RuntimeError from `generate_pseudo_masks.py` itself).
- **Val Dice throughout U-Net training** (Cell 17) is pseudo-mask Dice, not GT polygon Dice -- the GT comparison
  only happens once, in Cell 18, after every upstream decision (classifier checkpoint, pseudo-masks, U-Net
  checkpoint) is already frozen.
- Compare Cell 18's summary JSON against `thesis_final.ipynb`'s `unet_from_pseudo_summary.json` (same GT eval
  script, same metrics) to see whether anatomy-conditioning improved end-to-end tumor Dice/IoU over the
  standard `btxrd_best`/`btxrd_hybrid` pipeline.
